# GSM8K baselines on Colab

Greedy pass@1 on the GSM8K test set (1,319 problems) for every model in `common/models.py`.
Finished runs are skipped, so after a disconnect just run all cells again.

Works in the Colab web UI and in **VS Code with the Google Colab extension**.

**VS Code (Colab extension)**
1. Open this notebook, *Select Kernel > Colab > New Colab Server*, and pick a GPU machine (T4 is free; L4/A100 are much faster).
   The kernel runs on the Colab machine, not your laptop, so the setup cells clone the repo there.
2. Tokens: copy `.env.example` to `.env`, fill in `GITHUB_TOKEN` (fine-grained, read-only access to this repo), then right-click `.env` >
   *Upload to Colab* (it lands in `/content/.env`). Without it the clone cell prompts for the token.
   Or right-click the repo folder > *Upload to Colab*, and the clone is skipped.
3. Results: run *Colab: Mount Google Drive to Server...* from the command palette before the setup cell if you want them on Drive.
   Otherwise they are written to `/content/results` on the server. Download them from the Colab *Contents* view before removing the server.

**Colab web UI**
1. *Runtime > Change runtime type*: pick a GPU.
2. *Secrets* (key icon): add `GITHUB_TOKEN` (and optionally `HF_TOKEN`) with notebook access. Drive is mounted automatically.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup: Drive, code, packages, data

In [ ]:
import os

def load_env(paths=("/content/.env", "/content/implicit-adaptation/.env")):
    """Read KEY=VALUE lines from an uploaded .env into os.environ (existing vars win)."""
    for path in paths:
        if os.path.exists(path):
            for line in open(path):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    if v.strip():
                        os.environ.setdefault(k.strip(), v.strip().strip("'\""))
            print("loaded", path)

def secret(name):
    """Colab secret (web UI), else environment / uploaded .env, else None."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

load_env()

# Results go to Drive if it is (or can be) mounted, else to the server's local disk.
DRIVE = "/content/drive/MyDrive/implicit-adaptation"
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive not mounted ({type(e).__name__}); in VS Code use 'Colab: Mount Google Drive to Server...'")
BASE = DRIVE if os.path.isdir("/content/drive/MyDrive") else "/content"
RESULTS = f"{BASE}/results"   # full runs
SMOKE = f"{BASE}/smoke"       # 50-question checks
os.makedirs(RESULTS, exist_ok=True)
print("results ->", RESULTS)

if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")

In [ ]:
REPO = "/content/implicit-adaptation"
if os.path.exists(f"{REPO}/.git"):
    !git -C {REPO} pull -q
elif os.path.exists(f"{REPO}/evaluation"):
    print("using uploaded copy of the repo (no git pull)")
else:
    from getpass import getpass
    token = secret("GITHUB_TOKEN") or getpass("GitHub token (read access to the repo): ")
    !git clone -q https://{token}@github.com/ignoreandfly/implicit-adaptation.git {REPO}
%cd {REPO}
!git log --oneline -1 2>/dev/null || true

In [ ]:
!pip install -q -r requirements.txt
import torch, transformers
print(torch.__version__, transformers.__version__)

In [ ]:
if not os.path.exists("data/gsm8k/test.jsonl"):
    !python -m tasks.gsm8k --out data/gsm8k
!wc -l data/gsm8k/*.jsonl

## 2. Run list

Base models get the plain `Question:/Answer:` prompt (zero-shot and 8-shot); post-trained models get their chat template (zero-shot).
Batch sizes are set from the GPU; lower them if you hit out-of-memory.

In [ ]:
import json, glob

RUNS = [  # (registry name, few-shot exemplars)
    ("olmo2-1b-base", 0), ("olmo2-1b-base", 8),
    ("olmo2-1b-sft", 0), ("olmo2-1b-dpo", 0), ("olmo2-1b-instruct", 0),
]

bf16_gpu = torch.cuda.get_device_capability()[0] >= 8   # L4 / A100
BS_ZERO, BS_FEW = (64, 16) if bf16_gpu else (32, 8)     # 8-shot prompts are ~10x longer

def run(model, shots, root, limit=None):
    out = f"{root}/{model}/gsm8k/{shots}shot"
    if os.path.exists(f"{out}/test.metrics.json"):
        print("done already:", out)
        return
    bs = BS_ZERO if shots == 0 else BS_FEW
    lim = f"--limit {limit}" if limit else ""
    get_ipython().system(
        f'python -m evaluation.generate --model {model} --few_shot {shots} --batch_size {bs} {lim} --out "{out}"')

def show(root, n=2):
    """Print the first n outputs of every run under root."""
    for f in sorted(glob.glob(f"{root}/**/test.generations.jsonl", recursive=True)):
        print("=" * 100, "\n", f.replace(root + "/", ""))
        for line in open(f).readlines()[:n]:
            g = json.loads(line)
            s = g["scores"][0]
            print(f"--- gold {g['answer']} | strict {s['strict_answer']} | lenient {s['lenient_answer']} | {s['gen_tokens']} tok")
            print(g["outputs"][0][:1200])

def show_truncated(root, n=3):
    """Print outputs that hit max_new_tokens: loops, or answers that need more room?"""
    for f in sorted(glob.glob(f"{root}/**/test.generations.jsonl", recursive=True)):
        rows = [json.loads(l) for l in open(f)]
        cut = [g for g in rows if g["scores"][0]["truncated"]]
        print("=" * 100, "\n", f.replace(root + "/", ""), f"| {len(cut)}/{len(rows)} truncated")
        for g in cut[:n]:
            print("--- last 600 chars:\n", g["outputs"][0][-600:])

## 3. Smoke test: 50 questions per run

Takes a few minutes per model. Check the outputs by eye: do chat models end with "The answer is N"? Do base models stop cleanly? Is truncation near 0?

In [ ]:
for model, shots in RUNS:
    run(model, shots, SMOKE, limit=50)
!python -m analysis.summarize "{SMOKE}"

In [ ]:
show(SMOKE)

In [ ]:
show_truncated(SMOKE)

## 4. Full test set

Smoke timings on a T4 (50 problems): 60-90 s per zero-shot run, 180 s for 8-shot. The full set is 26x larger:
about 25-40 min per zero-shot run and 80 min for 8-shot, so about 3.5-4 h for all five on a T4 (an L4/A100 is several times faster).
Free Colab may disconnect; re-run the setup cells and this one, and finished runs are skipped.

In [ ]:
for model, shots in RUNS:
    run(model, shots, RESULTS)

In [ ]:
!python -m analysis.summarize "{RESULTS}"
print("table saved to", f"{RESULTS}/summary.csv")
if not RESULTS.startswith("/content/drive"):
    !cd /content && zip -qr results.zip results smoke && ls -lh /content/results.zip
    print("Not on Drive: download /content/results.zip from the Colab Contents view before removing the server.")